# 01 - Data exploration and cleaning

**Customer360 - Customer Retention & Churn Analytics**

Goal: understand the raw IBM Telco Customer Churn file, find every data-quality issue, and produce a documented clean dataset.
The raw file in `data/raw/` is never modified; all cleaning happens in `src/data_cleaning.py` and is written to `data/processed/`.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src import config, data_cleaning as dc, feature_engineering as fe, exploratory_analysis as eda
pd.set_option("display.width", 160, "display.max_columns", 40)
%matplotlib inline

## 1. Load the raw data and inspect its structure

In [2]:
raw = dc.load_raw_data()
profile = dc.inspect_dataset(raw)
print(f"{profile['n_rows']:,} rows x {profile['n_columns']} columns")
print("Duplicate rows:", profile["duplicate_rows"], "| duplicate customer IDs:", profile["duplicate_ids"])
print("NaN values:", profile["missing_values"] or "none", "| blank-string values:", profile["blank_strings"])
raw.head()

7,043 rows x 21 columns
Duplicate rows: 0 | duplicate customer IDs: 0
NaN values: none | blank-string values: {'TotalCharges': 11}


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
pd.DataFrame({"dtype": raw.dtypes.astype(str), "unique values": raw.nunique(), "example": raw.iloc[0]})

,dtype,unique values,example
customerID,str,7043,7590-VHVEG
gender,str,2,Female
SeniorCitizen,int64,2,0
Partner,str,2,Yes
Dependents,str,2,No
tenure,int64,73,1
PhoneService,str,2,No
MultipleLines,str,3,No phone service
InternetService,str,3,DSL
OnlineSecurity,str,3,No


**What the data shows.** 7,043 customers x 21 columns, no NaNs and no duplicates - but pandas reads `TotalCharges` as *text*,
which is a warning sign: the profile above reports **11 blank-string values** in that column. A plain `isna()` check would have missed them.

## 2. The `TotalCharges` blanks

In [4]:
blank = raw["TotalCharges"].astype("string").str.strip() == ""
raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Contract", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


**Finding.** All 11 customers have `tenure = 0` (they signed up but have not completed a billing period) and none has churned.
**Decision:** set `TotalCharges = 0.0` for them - that is the true amount billed. Dropping the rows would lose 11 genuine customers;
imputing a mean would invent revenue. (Any blank for a customer with tenure > 0 would be estimated as `MonthlyCharges x tenure`; none exist.)

## 3. Categorical consistency and invalid values

In [5]:
cat_cols = [c for c in raw.columns if raw[c].nunique() <= 5 and c != "Churn"]
for c in cat_cols:
    print(f"{c:18s}", raw[c].value_counts().to_dict())

gender             {'Male': 3555, 'Female': 3488}
SeniorCitizen      {0: 5901, 1: 1142}
Partner            {'No': 3641, 'Yes': 3402}
Dependents         {'No': 4933, 'Yes': 2110}
PhoneService       {'Yes': 6361, 'No': 682}
MultipleLines      {'No': 3390, 'Yes': 2971, 'No phone service': 682}
InternetService    {'Fiber optic': 3096, 'DSL': 2421, 'No': 1526}
OnlineSecurity     {'No': 3498, 'Yes': 2019, 'No internet service': 1526}
OnlineBackup       {'No': 3088, 'Yes': 2429, 'No internet service': 1526}
DeviceProtection   {'No': 3095, 'Yes': 2422, 'No internet service': 1526}
TechSupport        {'No': 3473, 'Yes': 2044, 'No internet service': 1526}
StreamingTV        {'No': 2810, 'Yes': 2707, 'No internet service': 1526}
StreamingMovies    {'No': 2785, 'Yes': 2732, 'No internet service': 1526}
Contract           {'Month-to-month': 3875, 'Two year': 1695, 'One year': 1473}
PaperlessBilling   {'Yes': 4171, 'No': 2872}
PaymentMethod      {'Electronic check': 2365, 'Mailed check': 1612, 'Bank

**Finding.** Labels are consistent (no case/spelling variants, no stray whitespace). Two redundancies exist:
`"No internet service"` repeats the information in `InternetService = "No"` across six add-on columns, and `"No phone service"`
duplicates `PhoneService = "No"`. **Decision:** map both to `"No"` so each service column is a clean Yes/No flag (the internet and phone
status stays available in its own column), relabel `InternetService = "No"` as `"No internet"`, and convert `SeniorCitizen` 0/1 to Yes/No.

In [6]:
num = raw.assign(TotalCharges=pd.to_numeric(raw["TotalCharges"], errors="coerce"))
print(num[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2))
expected = num["MonthlyCharges"] * num["tenure"]
rel = ((num["TotalCharges"] - expected).abs() / expected.where(num["tenure"] > 0)).dropna()
print(f"\nTotalCharges vs MonthlyCharges x tenure: median |diff| {rel.median():.1%}, within 10% for {(rel <= 0.10).mean():.0%} of customers")

        tenure  MonthlyCharges  TotalCharges
count  7043.00         7043.00       7032.00
mean     32.37           64.76       2283.30
std      24.56           30.09       2266.77
min       0.00           18.25         18.80
25%       9.00           35.50        401.45
50%      29.00           70.35       1397.48
75%      55.00           89.85       3794.74
max      72.00          118.75       8684.80

TotalCharges vs MonthlyCharges x tenure: median |diff| 2.0%, within 10% for 95% of customers


**Range checks.** Tenure runs 0-72 months, monthly charges $18.25-$118.75 (all positive), total charges are non-negative. `TotalCharges` is
close to `MonthlyCharges x tenure` for most customers; the differences are plausible (price changes, partial months), so **no values were altered or removed**.

## 4. Run the cleaning pipeline and validate

In [7]:
clean, log = dc.clean_data(raw)
print("Validation problems:", log.validation_problems or "none")
print(f"Clean dataset: {clean.shape[0]:,} rows x {clean.shape[1]} columns, missing values: {int(clean.isna().sum().sum())}")
display(pd.DataFrame({"raw dtype": [str(raw[c].dtype) for c in ["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]],
                      "clean dtype": [str(clean[c].dtype) for c in ["tenure_months", "monthly_charges", "total_charges", "senior_citizen"]]},
                     index=["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen"]))
print(log.total_charges)
print(log.categorical)

Validation problems: none
Clean dataset: 7,043 rows x 22 columns, missing values: 0


,raw dtype,clean dtype
tenure,int64,int64
MonthlyCharges,float64,float64
TotalCharges,str,float64
SeniorCitizen,int64,string


{'blank_total_charges': 11, 'set_to_zero_new_customers': 11, 'estimated_from_monthly_x_tenure': 0}
{'recoded_values': {'online_security': 1526, 'online_backup': 1526, 'device_protection': 1526, 'tech_support': 1526, 'streaming_tv': 1526, 'streaming_movies': 1526, 'multiple_lines': 682, 'internet_service': 1526}}


In [8]:
# Reconciliation: nothing was lost or changed besides the documented fixes
assert len(clean) == len(raw)
assert clean["churn_flag"].sum() == (raw["Churn"] == "Yes").sum()
assert np.isclose(clean["monthly_charges"].sum(), raw["MonthlyCharges"].sum())
assert np.isclose(clean["total_charges"].sum(), pd.to_numeric(raw["TotalCharges"], errors="coerce").sum())
print("Reconciled: row count, churn count, monthly-charge total and total-charge total match the raw file.")
clean.head()

Reconciled: row count, churn count, monthly-charge total and total-charge total match the raw file.


,customer_id,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract,paperless_billing,payment_method,monthly_charges,total_charges,churn,churn_flag
0,7590-VHVEG,Female,No,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,No,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


## 5. Engineered features and encoding

`src/feature_engineering.py` adds row-local features (tenure group, charge band, add-on counts, auto-pay flag, household flag).
They use only a customer's own columns - never the target - so they are safe to create before the train/test split.
Statistics-based encoding (one-hot, scaling) is done inside the model pipelines on training data only.

In [9]:
feats = fe.engineer_features(clean)
feats[["customer_id", "tenure_months", "tenure_group", "monthly_charges", "monthly_charge_band",
       "num_addon_services", "num_support_services", "is_autopay", "avg_monthly_spend", "has_household", "churn_flag"]].head(8)

,customer_id,tenure_months,tenure_group,monthly_charges,monthly_charge_band,num_addon_services,num_support_services,is_autopay,avg_monthly_spend,has_household,churn_flag
0,7590-VHVEG,1,0-12 months,29.85,<$35,1,1,No,29.85,Yes,0
1,5575-GNVDE,34,25-48 months,56.95,$35-70,2,2,No,55.57,No,0
2,3668-QPYBK,2,0-12 months,53.85,$35-70,2,2,No,54.08,No,1
3,7795-CFOCW,45,25-48 months,42.30,$35-70,3,3,Yes,40.91,No,0
4,9237-HQITU,2,0-12 months,70.70,$70-90,0,0,No,75.82,No,1
5,9305-CDSKC,8,0-12 months,99.65,$90+,3,1,No,102.56,No,1
6,1452-KIOVK,22,13-24 months,89.10,$70-90,2,1,Yes,88.61,Yes,0
7,6713-OKOMC,10,0-12 months,29.75,<$35,1,1,No,30.19,No,0


In [10]:
encoded = fe.encode_for_export(feats)
print("Fully numeric export:", encoded.shape, "| non-numeric columns:", [c for c in encoded.columns if c != "customer_id" and not pd.api.types.is_numeric_dtype(encoded[c])])

Fully numeric export: (7043, 27) | non-numeric columns: []


## 6. Files produced

| File | Content |
|---|---|
| `data/processed/telco_churn_clean.csv` | cleaned, readable labels, `churn_flag` target |
| `data/processed/telco_churn_features.csv` | clean + engineered features |
| `data/processed/telco_churn_encoded.csv` | fully numeric version for external tools |
| `reports/data_cleaning_report.md` | every decision with its rationale (generated) |

Next: `02_customer_churn_analysis.ipynb` - what drives churn.